# Headline letter accuracy, with honest error bars

Every earlier notebook reports one number per experiment. This one puts intervals on the
headline letter result and tests it against the baselines, using `evaluation.py` so later
notebooks report the same way.

**Protocol.** Leave-one-signer-out over the five dataset signers. Calibration rows (`user-*`)
are excluded from training and test. The model is the shipped recipe: 74 geometry features,
`StandardScaler`, `SVC(C=10, gamma="scale", class_weight="balanced")`.

**Why the intervals are wide.** There are five signers, so five independent test units. The
intervals resample signers, not holds.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import evaluation as ev

pd.set_option("display.width", 200)
P, M = ev.load_dataset()
F = ev.geometry_features(P)
y, signers = M.letter.to_numpy(), M.signer.to_numpy()
print(f"{len(y)} holds, {len(set(y))} letters, signers: {sorted(set(signers))}")
print("zero-variance features:", int((F.std(0) < 1e-9).sum()), "(d_0_9: palm length divided by itself)")

1345 holds, 40 letters, signers: ['signer1-F', 'signer2-F', 'signer3-F', 'signer4-L', 'signer5-R']
zero-variance features: 1 (d_0_9: palm length divided by itself)


## Fixed hyperparameters (the shipped model)

In [2]:
pred = ev.loso_predict(F, y, signers)
ok = (pred == y).astype(float)
maj = ev.majority_correct(y, signers)

folds = pd.DataFrame({"holds": pd.Series(signers).value_counts().sort_index(),
                      "letters": pd.Series(y).groupby(signers).nunique(),
                      "accuracy %": 100 * ev.per_fold(ok, signers),
                      "majority %": 100 * ev.per_fold(maj, signers)})
print(folds.round(1).to_string())
row = ev.summarise("letter accuracy, C=10 gamma=scale", ok, signers, baseline=maj)
print()
print(pd.DataFrame([row]).round(4).T.to_string(header=False))
recall = pd.Series(ok).groupby(y).mean()
print(f"\nmacro recall over 40 letters: {100*recall.mean():.1f}%   letters with recall 0: {(recall == 0).sum()}")

           holds  letters  accuracy %  majority %
signer1-F    311       40        21.9         9.6
signer2-F    302       40        32.5         9.9
signer3-F    279       39        29.0        10.4
signer4-L    295       38        22.0        10.2
signer5-R    158       36        27.2         9.5

metric             letter accuracy, C=10 gamma=scale
n                                               1345
pooled %                                     26.3941
mean of folds %                              26.5193
t half-width                                  5.6818
boot lo %                                    22.5547
boot hi %                                    30.3788
vs baseline (pts)                            16.5918
paired t p                                    0.0012

macro recall over 40 letters: 23.7%   letters with recall 0: 6


## Nested tuning

`C` and `gamma` above were chosen by hand. Here they are selected inside the four training
signers (an inner leave-one-signer-out), so the held-out signer is used once.

In [3]:
GRID = [(C, g) for C in (1, 10, 100) for g in ("scale", 0.01, 0.1)]

def nested_predict(F, y, signers):
    pred = np.empty(len(y), dtype=object); chosen = {}
    for s in sorted(set(signers)):
        tr, te = signers != s, signers == s
        best = None
        for C, g in GRID:
            p = ev.loso_predict(F[tr], y[tr], signers[tr], lambda: ev.svm(C, g))
            mu = ev.per_fold((p == y[tr]).astype(float), signers[tr]).mean()
            if best is None or mu > best[0]:
                best = (mu, C, g)
        _, C, g = best
        chosen[s] = f"C={C}, gamma={g}"
        pred[te] = ev.svm(C, g).fit(F[tr], y[tr]).predict(F[te])
    return pred, chosen

pred_n, chosen = nested_predict(F, y, signers)
ok_n = (pred_n == y).astype(float)
print(pd.DataFrame({"chosen on training signers": chosen,
                    "accuracy %": (100 * ev.per_fold(ok_n, signers)).round(1)}).to_string())
row_n = ev.summarise("letter accuracy, nested-tuned", ok_n, signers, baseline=maj)
print()
print(pd.DataFrame([row, row_n]).round(3).to_string(index=False))

          chosen on training signers  accuracy %
signer1-F           C=10, gamma=0.01        22.5
signer2-F          C=10, gamma=scale        32.5
signer3-F           C=1, gamma=scale        25.1
signer4-L          C=10, gamma=scale        22.0
signer5-R          C=10, gamma=scale        27.2

                           metric    n  pooled %  mean of folds %  t half-width  boot lo %  boot hi %  vs baseline (pts)  paired t p
letter accuracy, C=10 gamma=scale 1345    26.394           26.519         5.682     22.555     30.379             16.592       0.001
    letter accuracy, nested-tuned 1345    25.725           25.859         5.258     22.747     29.600             15.932       0.001


## Is it above chance?

Labels are shuffled within each signer and the whole leave-one-signer-out run is repeated.
That is what this pipeline scores when the features carry no letter information.

In [4]:
null = ev.permutation_null(F, y, signers, runs=30, seed=0)
obs = ev.per_fold(ok, signers).mean()
print(f"shuffled-label null over {len(null)} runs: mean {100*null.mean():.1f}%, max {100*null.max():.1f}%")
print(f"observed: {100*obs:.1f}%   uniform chance over 40 letters: 2.5%")
print(f"runs at or above observed: {(null >= obs).sum()} of {len(null)}")

shuffled-label null over 30 runs: mean 3.9%, max 4.9%
observed: 26.5%   uniform chance over 40 letters: 2.5%
runs at or above observed: 0 of 30


## Does the seed matter?

The SVM itself is deterministic, so letter accuracy has no seed variance: the only spread is
between signers. Seeds do matter for anything using `probability=True` (Platt scaling fits on
internal random folds). That affects word decoding, which `end_to_end_words.ipynb` reports
over several seeds.

In [5]:
accs = []
for seed in range(5):
    p = ev.loso_predict(F, y, signers, lambda: ev.svm(seed=seed))
    accs.append(100 * ev.per_fold((p == y).astype(float), signers).mean())
print("mean-of-folds letter accuracy over seeds 0-4:", np.round(accs, 2))

mean-of-folds letter accuracy over seeds 0-4: [26.52 26.52 26.52 26.52 26.52]


## What to quote

* **Letter accuracy on an unseen signer: 25.9% nested-tuned (26.4% pooled with the shipped
  settings).** Interval from resampling signers: see the table above.
* The majority baseline is 10.0%; the gap is significant on a paired test over the five folds,
  and the shuffled-label null never comes near the observed score.
* Macro recall is lower than accuracy, and six letters are never recognised. Those six have
  4 to 8 examples each.
* "Accuracy" here means agreement with pseudo-labels that no PSL reader has checked.